# Cold-Chain Shipment Delay Prediction

**AI Services MS2 | Supplementary Practice A | 60 Marks | Exactly 20 Evaluation Checks**

**Instructor reference notebook — complete working code.**

**Business task.** A cold-chain logistics company transports temperature-sensitive parcels between regions. Dispatch records include temperature, estimated transit duration, and a shipment note written by an operations agent. The distance and parcel weight exist inside the note rather than in ready-to-use numeric columns.

**Objective.** Build a reusable binary-classification notebook. Clean mixed-format dates and inconsistent category labels, extract two decimal measurements from shipment notes with simple regular expressions, engineer safe numerical features, and compare Logistic Regression with Random Forest. Use a probability threshold of 0.40 to label risky shipments.

## 1. Imports, parameters and feature groups

In [ ]:
import re

import joblib

import numpy as np

import pandas as pd

from sklearn.base import BaseEstimator

from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

TARGET = "late_delivery"

RAW_NUMERIC = ["transit_hours", "temperature_c"]

NUMERIC = ["transit_hours", "temperature_c", "distance_km", "weight_kg",
           "month_number", "departure_hour", "distance_per_hour"]

CATEGORICAL = ["region", "vehicle_type", "route_zone"]

FEATURES = NUMERIC + CATEGORICAL

MONTH_MAP = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5,
             "jun": 6, "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12}

THRESHOLD = 0.40

In [ ]:
def extract_decimal(text: object, field: str) -> float:
    """Return float parsed from simple 'distance: 12.5' or 'weight: 3.2' text; otherwise NaN."""
    match = re.search(rf"{field}\s*:\s*(\d+(?:\.\d+)?)", str(text), flags=re.I)
    
    return float(match.group(1)) if match else np.nan

def load_data(filename: str) -> pd.DataFrame:
    """Clean a shipment CSV and preserve valid input order, IDs and optional target."""

    df = pd.read_csv(filename, dtype={"shipment_id": "string"})
    df = df.drop_duplicates("shipment_id", keep="first").copy()
    dates = pd.to_datetime(df["dispatch_ts"], format="mixed", dayfirst=True, errors="coerce")

    df = df.loc[dates.notna()].copy()
    # df = df.dropna(subset=[dates.name]).copy()

    df["dispatch_ts"] = dates.loc[df.index]

    for col in RAW_NUMERIC:
        df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    
    for col in CATEGORICAL:
        df[col] = (df[col].astype("string").str.strip().str.lower()
                    .replace("", np.nan)
                    .str.replace(r"^([+-]?\d+)\.0+$", r"\1", regex=True)
                    .fillna("unknown").astype(str))
        
        
    df["distance_km"] = df["shipment_note"].apply(lambda x: extract_decimal(x, "distance"))
    df["weight_kg"] = df["shipment_note"].apply(lambda x: extract_decimal(x, "weight"))

    df["month_number"] = df["month_text"].astype("string").str.strip().str.lower().map(MONTH_MAP)
    df["departure_hour"] = df["dispatch_ts"].dt.hour

    speed_time = df["transit_hours"].where(df["transit_hours"] > 0)


    df["distance_per_hour"] = (df["distance_km"] / speed_time).replace([np.inf, -np.inf], np.nan)
    return df.reset_index(drop=True)

###########################################################################################

def build_pipeline(algorithm: BaseEstimator) -> Pipeline:
    numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),
                                ("scaler", StandardScaler())])
    
    categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                    ("encoder", OneHotEncoder(handle_unknown="ignore"))])
    
    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, NUMERIC),
        ("category", categorical_pipeline, CATEGORICAL),
    ])

    return Pipeline([("preprocessor", preprocessor), ("model", algorithm)])

####################################################################################3

def prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    
    
    train_df, test_df = load_data(train_file), load_data(test_file)
    
    for df in (train_df, test_df):
        labels = df[TARGET].astype("string").str.strip().str.lower()
        df[TARGET] = labels.map({"yes": 1, "no": 0, "1": 1, "0": 0})
        df.dropna(subset=[TARGET], inplace=True)
    
    return (train_df[FEATURES], test_df[FEATURES],
            train_df[TARGET].astype(int), test_df[TARGET].astype(int))


#####################################################################################


def train_models(X_train: pd.DataFrame, y_train: pd.Series) -> dict[str, Pipeline]:
    
    models = {
        "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
        "random_forest": RandomForestClassifier(n_estimators=80, max_depth=9,
            min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1),
    }
    trained_models = {}
    for name, algorithm in models.items():
        trained_models[name] = build_pipeline(algorithm).fit(X_train, y_train)
    
    return trained_models


##################################################################################


def evaluate_models(trained_models: dict[str, Pipeline], X_test: pd.DataFrame, y_test: pd.Series) -> pd.DataFrame:
    
    results = []
    for name, pipeline in trained_models.items():
        positive_idx = list(pipeline.classes_).index(1)
        probability = pipeline.predict_proba(X_test)[:, positive_idx]
        predicted = (probability >= THRESHOLD).astype(int)
        results.append({"model_name": name,
                        "accuracy": accuracy_score(y_test, predicted),
                        "precision": precision_score(y_test, predicted, zero_division=0),
                        "recall": recall_score(y_test, predicted, zero_division=0),
                        "f1": f1_score(y_test, predicted, zero_division=0)})
    return pd.DataFrame(results).sort_values(
        ["f1", "recall", "model_name"], ascending=[False, False, True]
    ).reset_index(drop=True)

####################################################################################

def select_best_model(comparison: pd.DataFrame, trained_models: dict[str, Pipeline]) -> tuple[str, Pipeline]:
    
    best_name = str(comparison.iloc[0]["model_name"])
    return best_name, trained_models[best_name]

#########################################################################################

def predict_delay(model: Pipeline, df: pd.DataFrame) -> pd.DataFrame:
    
    positive_idx = list(model.classes_).index(1)
    probabilities = model.predict_proba(df[FEATURES])[:, positive_idx]

    return pd.DataFrame({"shipment_id": df["shipment_id"].to_numpy(),
                        "predicted_late": (probabilities >= THRESHOLD).astype(int),
                        "late_probability": np.round(probabilities, 4)})

In [ ]:
def train() -> tuple[str, pd.DataFrame]:
    
    X_train, X_test, y_train, y_test = prepare_datasets("shipment_train.csv", "shipment_test.csv")
    trained = train_models(X_train, y_train)
    comparison = evaluate_models(trained, X_test, y_test)
    name, model = select_best_model(comparison, trained)
    joblib.dump(model, "best_model.pkl")
    comparison.to_csv("model_comparison.csv", index=False)
    return name, comparison

def predict() -> pd.DataFrame:
    
    df = load_data("shipment_predict.csv")
    model = joblib.load("best_model.pkl")
    result = predict_delay(model, df)
    result.to_csv("shipment_predictions.csv", index=False)
    return result

## 4. Run the workflow

Execute the **training** call first, then the **prediction** call. You can also run these in separate notebook sessions after a model has been saved.

In [ ]:
best_name, comparison = train()
display(comparison)

In [ ]:
result = predict()
display(result.head())